# 01 — Raw EDA: Sri Lankan Cinnamon Export Sales

Spec: `specs/003-complete-forecasting-submission/spec.md` FR-1–FR-5.

This notebook explores the **raw, untouched** export workbook (`data/raw/Cinnamon_export_sales.xlsx`)
before any cleaning or scoping is applied, and separately demonstrates the cleaning/scoping stages
by calling the shared `src/` logic (never re-implementing business rules inline, per FR-4).

All counts below are computed live against the current raw file — nothing is hard-coded.

In [1]:
import sys
from pathlib import Path

# Ensure repo root is on sys.path so `from src...` imports resolve regardless of the
# notebook's working directory (nbconvert executes with cwd = notebooks/).
_repo_root = Path.cwd()
if not (_repo_root / "src").exists():
    _repo_root = _repo_root.parent
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))
import os
os.chdir(_repo_root)

import pandas as pd
import matplotlib.pyplot as plt

from src.cleaning import (
    RAW_PATH,
    ARTIFACT_INDICES,
    _read_raw,
    clean_transactions,
)
from src.scoping import (
    MERCHANDISE_RANGES,
    RUBBER_LATEX_RANGES,
    EXCLUDED_PRODUCT_RANGES,
    scope_products,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

raw = _read_raw(RAW_PATH)
print(f"Loaded raw workbook: {RAW_PATH}")
print(f"Raw row count: {len(raw):,}")

Loaded raw workbook: data/raw/Cinnamon_export_sales.xlsx
Raw row count: 60,670


## 1. Raw row count, columns, dtypes

In [2]:
print(f"Rows: {len(raw):,}")
print(f"Columns ({len(raw.columns)}): {list(raw.columns)}")
print()
print("Dtypes:")
print(raw.dtypes)

Rows: 60,670
Columns (14): ['Region', 'Country', 'Customer Code', 'Customer ID', 'Brand Category', 'Product Range', 'Sales Channel', 'Product Code', 'Order Date', 'Invoice Date', 'Invoice No', 'Sales USD', 'Sales Qty', 'Sales KG']

Dtypes:
Region                str
Country               str
Customer Code       int64
Customer ID           str
Brand Category        str
Product Range         str
Sales Channel         str
Product Code          str
Order Date            str
Invoice Date          str
Invoice No            str
Sales USD         float64
Sales Qty         float64
Sales KG          float64
dtype: object


## 2. Date range: Order Date / Invoice Date

In [3]:
order_dt = pd.to_datetime(raw["Order Date"], errors="coerce")
invoice_dt = pd.to_datetime(raw["Invoice Date"], errors="coerce")

print(f"Order Date   range: {order_dt.min()} .. {order_dt.max()}  (missing: {order_dt.isna().sum():,})")
print(f"Invoice Date range: {invoice_dt.min()} .. {invoice_dt.max()}  (missing: {invoice_dt.isna().sum():,})")

Order Date   range: 2022-02-28 00:00:00 .. 2025-09-17 00:00:00  (missing: 142)
Invoice Date range: 2022-04-01 00:00:00 .. 2025-09-16 00:00:00  (missing: 3)


## 3. Region / Country counts — assignment-brief discrepancy (FR-3)

In [4]:
raw_no_artifacts = raw.drop(index=raw.index.intersection(ARTIFACT_INDICES))

region_counts = raw_no_artifacts["Region"].value_counts(dropna=True)
country_counts = raw_no_artifacts["Country"].value_counts(dropna=True)

n_regions = raw_no_artifacts["Region"].nunique(dropna=True)
n_countries = raw_no_artifacts["Country"].nunique(dropna=True)

print(f"Distinct Region values (excl. the 3 export-artifact rows): {n_regions}")
print(region_counts)
print()
print(f"Distinct Country values: {n_countries}")
print()
print(
    f"DISCREPANCY (FR-3): the assignment brief states 19 regions; "
    f"verified data has {n_regions} real regions (the raw file's naive nunique() reads 19 only "
    f"because it counts the 'Total' pivot row and the 'Applied filters:' footer row in the Region "
    f"column as if they were region names; excluding the 3 known export-artifact rows gives the "
    f"correct {n_regions}-region count)."
)


Distinct Region values (excl. the 3 export-artifact rows): 17
Region
ASIA                   19454
EUROPE                 14444
OCEANIA                 9692
MIDDLE EAST             8142
CIS                     3087
AFRICA                  2480
SOUTH AMERICA           2022
NORTH AMERICA           1069
WEST INDIES              162
WESTERN EUROPE            61
EAST ASIA                 33
LOCAL MARKET               9
SOUTH ASIA                 4
SOUTHEAST ASIA             3
NORTH AFRICA               2
CENTRAL EUROPE             2
SOUTHEASTERN EUROPE        1
Name: count, dtype: int64

Distinct Country values: 91

DISCREPANCY (FR-3): the assignment brief states 19 regions; verified data has 17 real regions (the raw file's naive nunique() reads 19 only because it counts the 'Total' pivot row and the 'Applied filters:' footer row in the Region column as if they were region names; excluding the 3 known export-artifact rows gives the correct 17-region count).


## 4. Distinct Product Code / derived ProductID counts

`ProductID` is derived by `src.cleaning.clean_transactions()` as the first 9 characters of
`Product Code` — that business rule lives in `src/cleaning.py` and is reused here, not
reimplemented.

In [5]:
n_product_codes_raw = raw["Product Code"].nunique(dropna=True)
print(f"Distinct raw 'Product Code' values: {n_product_codes_raw:,}")

# Reuse the shared cleaning pipeline (Stage 2) to get the derived ProductID column.
cleaned = clean_transactions(raw_path=RAW_PATH)
n_product_ids = cleaned["ProductID"].nunique(dropna=True)
print(f"Distinct derived 'ProductID' values (post-clean, 9-char prefix): {n_product_ids:,}")
print()
print(f"Cleaned row count (post artifact-drop + dedup): {len(cleaned):,}")
print(f"Artifact rows dropped: {cleaned.attrs['artifact_rows_dropped']}")
print(f"Duplicate rows dropped (post-artifact dedup): {cleaned.attrs['duplicate_rows_dropped']}")
print(f"Missing Order Date rows imputed: {cleaned.attrs['missing_order_dates_imputed']}")

Distinct raw 'Product Code' values: 13,734


Distinct derived 'ProductID' values (post-clean, 9-char prefix): 13,724

Cleaned row count (post artifact-drop + dedup): 60,646
Artifact rows dropped: 3
Duplicate rows dropped (post-artifact dedup): 21
Missing Order Date rows imputed: 139


## 5. Missing values, duplicates, negative quantities/revenue

Missing-value counts are reported on the **raw** workbook (all 60,670 rows, including the
3 export-artifact rows). Duplicate-row count is reported both on the full raw file and on the
post-artifact-removal file (the number `clean_transactions()` actually drops as duplicates).

In [6]:
print("Missing values per column (raw, all rows):")
print(raw.isna().sum())
print()

full_raw_duplicates = int(raw.duplicated().sum())
print(f"Duplicate rows in the FULL uncleaned file (incl. artifact rows): {full_raw_duplicates}")
print(f"Duplicate rows dropped by clean_transactions() (post-artifact removal): "
      f"{cleaned.attrs['duplicate_rows_dropped']}")
print()

neg_qty = int((raw["Sales Qty"] < 0).sum())
neg_usd = int((raw["Sales USD"] < 0).sum())
print(f"Rows with negative Sales Qty: {neg_qty}")
print(f"Rows with negative Sales USD: {neg_usd}")

Missing values per column (raw, all rows):
Region              1
Country             3
Customer Code       0
Customer ID         0
Brand Category      3
Product Range       3
Sales Channel       3
Product Code        0
Order Date        142
Invoice Date        3
Invoice No          0
Sales USD           2
Sales Qty           2
Sales KG            2
dtype: int64

Duplicate rows in the FULL uncleaned file (incl. artifact rows): 21
Duplicate rows dropped by clean_transactions() (post-artifact removal): 21

Rows with negative Sales Qty: 140
Rows with negative Sales USD: 103


## 6. The 3 export-artifact rows

`src.cleaning.ARTIFACT_INDICES` (`60667, 60668, 60669`) marks 3 rows appended by the export tool
that are not real transactions. `clean_transactions()` drops them via `_drop_export_artifacts()`
before any other processing.

In [7]:
artifact_rows = raw.loc[raw.index.intersection(ARTIFACT_INDICES)]
display(artifact_rows)

print()
print("Row-by-row explanation:")
print(f"  index {ARTIFACT_INDICES[0]}: pivot 'Total' row (Region == 'Total') — an aggregate summary row, not a transaction.")
print(f"  index {ARTIFACT_INDICES[1]}: blank spacer row (all key fields null) inserted by the export tool.")
print(f"  index {ARTIFACT_INDICES[2]}: 'Applied filters:' footer row describing the export's filter settings, not data.")

,Region,Country,Customer Code,Customer ID,Brand Category,Product Range,Sales Channel,Product Code,Order Date,Invoice Date,Invoice No,Sales USD,Sales Qty,Sales KG
60667,Total,NaN,73883,C2,NaN,NaN,NaN,43962-015-R07-ZOF-EJM-0148ST-3.0G-STD,NaN,NaN,INV/71718263,222396655.0,14115419.0,17635059.0
60668,NaN,NaN,73883,C2,NaN,NaN,NaN,43962-015-R07-ZOF-EJM-0148ST-3.0G-STD,NaN,NaN,INV/71718263,NaN,NaN,NaN
60669,"Applied filters:\nFinancial Year is 2023/2024,...",NaN,73883,C2,NaN,NaN,NaN,43962-015-R07-ZOF-EJM-0148ST-3.0G-STD,NaN,NaN,INV/71718263,NaN,NaN,NaN



Row-by-row explanation:
  index 60667: pivot 'Total' row (Region == 'Total') — an aggregate summary row, not a transaction.
  index 60668: blank spacer row (all key fields null) inserted by the export tool.
  index 60669: 'Applied filters:' footer row describing the export's filter settings, not data.


## 7. Merchandise / rubber-latex exclusion categories (FR-4)

Exclusion lists are imported directly from `src.scoping` (`MERCHANDISE_RANGES`,
`RUBBER_LATEX_RANGES`) — not duplicated here. We report how many raw/cleaned rows fall into
each excluded category, then call the shared `scope_products()` stage to get the actual
in-scope row/product counts.

In [8]:
print(f"MERCHANDISE_RANGES ({len(MERCHANDISE_RANGES)} categories):")
for r in sorted(MERCHANDISE_RANGES):
    print(f"  - {r}")
print()
print(f"RUBBER_LATEX_RANGES ({len(RUBBER_LATEX_RANGES)} categories):")
for r in sorted(RUBBER_LATEX_RANGES):
    print(f"  - {r}")

product_range = cleaned["Product Range"].astype("string").str.strip().str.upper()
merch_count = int(product_range.isin(MERCHANDISE_RANGES).sum())
rubber_count = int(product_range.isin(RUBBER_LATEX_RANGES).sum())
excluded_count = int(product_range.isin(EXCLUDED_PRODUCT_RANGES).sum())

print()
print(f"Cleaned rows matching a MERCHANDISE category: {merch_count:,}")
print(f"Cleaned rows matching a RUBBER/LATEX category: {rubber_count:,}")
print(f"Cleaned rows matching any excluded category:   {excluded_count:,}")

MERCHANDISE_RANGES (8 categories):
  - APPAREL
  - BROCHURES / MAGAZINE
  - GLASS / CERAMIC/ PORCELAIN
  - PACKAGING
  - POINT OF SALE MATERIALS
  - PROMOTIONAL MATERIAL
  - PVC/PRESENTERS
  - WOODEN BOXES / PRESENTERS

RUBBER_LATEX_RANGES (8 categories):
  - BULK RUBBER
  - CENTRIFUGED LATEX
  - PALE CREPE
  - PREMIUM CREPE GRADE
  - PREMIUM LATEX GRADE
  - PROCESSED SHEETS
  - WATTE REGIONAL GRADE
  - WATTE SINGLE ESTATE GRADE

Cleaned rows matching a MERCHANDISE category: 6,542
Cleaned rows matching a RUBBER/LATEX category: 3,109
Cleaned rows matching any excluded category:   9,651


In [9]:
# Reuse the shared Stage 3 scoping logic (src.scoping.scope_products) — do not
# reimplement the exclusion filter inline.
from src.cleaning import CLEANED_PATH

scoped = scope_products(cleaned_path=CLEANED_PATH)

print(f"Cleaned rows (pre-scope):   {scoped.attrs['cleaned_rows']:,}")
print(f"Excluded rows (merch+rubber): {scoped.attrs['excluded_rows']:,}")
print(f"In-scope rows (post-scope): {scoped.attrs['scoped_rows']:,}")
print()
print(f"Cleaned products (pre-scope): {scoped.attrs['cleaned_products']:,}")
print(f"In-scope products (post-scope): {scoped.attrs['scoped_products']:,}")

Cleaned rows (pre-scope):   60,646
Excluded rows (merch+rubber): 9,651
In-scope rows (post-scope): 50,995

Cleaned products (pre-scope): 13,724
In-scope products (post-scope): 9,052


## Summary of verified counts

This cell recomputes and prints the headline reference numbers cited throughout the
spec 001/003 documents, straight from the objects computed above (no hard-coded values).

In [10]:
summary = {
    "raw_rows": len(raw),
    "raw_regions": n_regions,
    "raw_countries": n_countries,
    "cleaned_rows_post_dedup": len(cleaned),
    "duplicate_rows_dropped": cleaned.attrs["duplicate_rows_dropped"],
    "missing_order_date_rows_imputed": cleaned.attrs["missing_order_dates_imputed"],
    "negative_sales_qty_rows": neg_qty,
    "negative_sales_usd_rows": neg_usd,
    "cleaned_products": scoped.attrs["cleaned_products"],
    "in_scope_rows": scoped.attrs["scoped_rows"],
    "in_scope_products": scoped.attrs["scoped_products"],
}
for k, v in summary.items():
    print(f"{k}: {v:,}" if isinstance(v, int) else f"{k}: {v}")

raw_rows: 60,670
raw_regions: 17
raw_countries: 91
cleaned_rows_post_dedup: 60,646
duplicate_rows_dropped: 21
missing_order_date_rows_imputed: 139
negative_sales_qty_rows: 140
negative_sales_usd_rows: 103
cleaned_products: 13,724
in_scope_rows: 50,995
in_scope_products: 9,052
